In [42]:
# ============================================================
# ЯЧЕЙКА 1: Импорты и исходные данные
# ============================================================
import json
import csv
import os
from collections import Counter

# Схемы таблиц
Commission = dict(id=int, name=str, code=str, specialization=int)
Specialization = dict(id=int, code=str)
SpecializationInCommission = dict(id=int, commission=int, specialization=int)
Scientist = dict(id=int, name=str, commission=int, specialization=int)
Publication = dict(id=int, title=str, scientist=int, journal=int)
Journal = dict(id=int, title=str)
JournalSpecialization = dict(journal=int, specialization=int)
ScientistInCommission = dict(scientist=int, commission=int, specialization=int)

# Таблицы данных
CommissionTable = [
    (1, "Совет по математике", "МАТ-01", 101),
    (2, "Совет по физике", "ФИЗ-02", 102),
    (3, "Совет по информатике", "ИНФ-03", 103),
]

SpecializationTable = [
    (101, "01.01.01"),
    (102, "01.04.02"),
    (103, "05.13.11"),
]

SpecializationInCommissionTable = [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 102),
    (4, 3, 103),
]

ScientistTable = [
    (1, "Иванов И.И.", 1, 101),
    (2, "Петров П.П.", 1, 102),
    (3, "Сидоров С.С.", 2, 102),
    (4, "Кузнецова А.А.", 3, 103),
]

PublicationTable = [
    (1, "Математические модели", 1, 10),
    (2, "Квантовая физика", 2, 11),
    (3, "Искусственный интеллект", 4, 12),
    (4, "Численные методы", 1, 13),
]

JournalTable = [
    (10, "Вестник РАН"),
    (11, "ЖЭТФ"),
    (12, "AI Journal"),
    (13, "Выч. математика"),
]

JournalSpecializationTable = [
    (10, 101),
    (11, 102),
    (12, 103),
    (13, 101),
]

ScientistInCommissionTable = [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 102),
    (4, 3, 103),
]

# Множество источников S
S = [
    ("Commission", Commission, CommissionTable),
    ("Specialization", Specialization, SpecializationTable),
    ("SpecializationInCommission", SpecializationInCommission, SpecializationInCommissionTable),
    ("Scientist", Scientist, ScientistTable),
    ("Publication", Publication, PublicationTable),
    ("Journal", Journal, JournalTable),
    ("JournalSpecialization", JournalSpecialization, JournalSpecializationTable),
    ("ScientistInCommission", ScientistInCommission, ScientistInCommissionTable),
]

In [43]:
# ============================================================
# ЯЧЕЙКА 2 (финальная): Правила БЕЗ subject_source и object_source
# ============================================================
rules_json = {
    "ontology": {
        "name": "Онтология академической среды",
        "classes": ["Учёный", "Публикация", "Журнал", "Специальность", "Диссовет"],
        "relations": [
            "фио", "название", "шифр", "код",
            "опубликовал_статью", "опубликована_в",
            "покрывает_специальность", "включает_специальность",
            "состоит_в_совете", "представляет_специальность"
        ]
    },
    "rules": {
        "structured": [
            # === Scientist: создаём субъект ===
            {"source": "Scientist", "subject_field": "id", "relation": "фио", "object_field": "name", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Publication: создаём субъект ===
            {"source": "Publication", "subject_field": "id", "relation": "название", "object_field": "title", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Связь: Scientist --опубликовал_статью--> Publication ===
            {"source": "Publication", "subject_field": "scientist", "relation": "опубликовал_статью", "object_field": "id", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Publication", "subject_field": "id", "relation": "опубликована_в", "object_field": "journal", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Journal: создаём субъект ===
            {"source": "Journal", "subject_field": "id", "relation": "название", "object_field": "title", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Связь: Journal --покрывает_специальность--> Specialization ===
            {"source": "JournalSpecialization", "subject_field": "journal", "relation": "покрывает_специальность", "object_field": "specialization", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Commission: создаём субъект ===
            {"source": "Commission", "subject_field": "id", "relation": "название", "object_field": "name", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "Commission", "subject_field": "id", "relation": "шифр", "object_field": "code", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Specialization: создаём субъект ===
            {"source": "Specialization", "subject_field": "id", "relation": "код", "object_field": "code", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Связь: Commission --включает_специальность--> Specialization ===
            {"source": "SpecializationInCommission", "subject_field": "commission", "relation": "включает_специальность", "object_field": "specialization", "time_default": "2025-01-01", "confidence": 1.0},
            
            # === Связи: Scientist --состоит_в_совете--> Commission, Scientist --представляет_специальность--> Specialization ===
            {"source": "ScientistInCommission", "subject_field": "scientist", "relation": "состоит_в_совете", "object_field": "commission", "time_default": "2025-01-01", "confidence": 1.0},
            {"source": "ScientistInCommission", "subject_field": "scientist", "relation": "представляет_специальность", "object_field": "specialization", "time_default": "2025-01-01", "confidence": 1.0}
        ],
        "semi_structured": [],
        "unstructured": []
    }
}

with open("ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)
print("✓ ontology_rules.json сохранён")

✓ ontology_rules.json сохранён


In [44]:
# ============================================================
# ЯЧЕЙКА 3 (финальная v4): apply_rule с префиксами и у объектов
# ============================================================
def apply_rule(source_name, rule, data_sources):
    """
    Применяет правило.
    Тип субъекта и объекта определяется автоматически по имени поля:
    - 'id' → сущность = source_name
    - внешний ключ (scientist, journal, commission, specialization) → сущность = имя поля
    """
    facts = []
    
    if source_name not in data_sources:
        print(f"  ВНИМАНИЕ: Источник {source_name} не найден")
        return facts
    
    schema, table = data_sources[source_name]
    columns = list(schema.keys())
    
    subject_field = rule.get("subject_field")
    object_field = rule.get("object_field")
    object_value = rule.get("object_value")
    time_field = rule.get("time_field")
    time_default = rule.get("time_default", "2025-01-01")
    value_field = rule.get("value_field")
    confidence = rule.get("confidence", 1.0)
    relation = rule["relation"]
    
    for row in table:
        row_dict = dict(zip(columns, row))
        
        # === СУБЪЕКТ ===
        if subject_field is not None and subject_field in row_dict:
            if subject_field == "id":
                # ID таблицы → Publication_1
                subject = f"{source_name}_{row_dict[subject_field]}"
            else:
                # Внешний ключ: scientist → Scientist_1
                entity = subject_field[0].upper() + subject_field[1:]
                subject = f"{entity}_{row_dict[subject_field]}"
        else:
            subject = None
        
        # === ОБЪЕКТ ===
        if object_value is not None:
            # Константа: "Учёный", "Журнал"
            obj = object_value
        elif object_field is not None and object_field in row_dict:
            if object_field == "id":
                # ID таблицы → Publication_1
                obj = f"{source_name}_{row_dict[object_field]}"
            elif object_field in ["name", "title", "code", "year"]:
                # Строковое поле → значение как есть
                obj = str(row_dict[object_field])
            else:
                # Внешний ключ: journal → Journal_10
                entity = object_field[0].upper() + object_field[1:]
                obj = f"{entity}_{row_dict[object_field]}"
        else:
            obj = None
        
        # === ВРЕМЯ ===
        if time_field is not None and time_field in row_dict:
            time_val = str(row_dict[time_field])
        else:
            time_val = time_default
        
        # === ЗНАЧЕНИЕ ===
        n_value = row_dict[value_field] if (value_field is not None and value_field in row_dict) else None
        
        facts.append((subject, relation, obj, time_val, n_value, confidence))
    
    return facts

In [45]:
# ============================================================
# ЯЧЕЙКА 4: Запуск интеграции
# ============================================================
print("=" * 60)
print("ОНТОЛОГИЧЕСКАЯ ИНТЕГРАЦИЯ ДАННЫХ О ДИССОВЕТАХ")
print("=" * 60)

# Проверяем S
print(f"\nИсточников данных: {len(S)}")
for name, schema, table in S:
    print(f"  {name}: {len(table)} записей (колонки: {', '.join(schema.keys())})")

# Запуск интеграции
facts, stats, ontology = integrate(S)

print(f"\nВсего фактов: {len(facts)}")
print(f"\nРаспределение по отношениям:")
for rel, count in sorted(stats.items()):
    print(f"  {rel}: {count}")

# Проверка покрытия
expected = set(ontology["relations"])
actual = set(stats.keys())
if expected - actual:
    print(f"\nВНИМАНИЕ: Отсутствуют отношения: {expected - actual}")
else:
    print(f"\n✓ Покрытие онтологии: 100% ({len(actual)}/{len(expected)} отношений)")

# Сохранение
save_to_csv(facts, "experiment_data_soviets/generated/all_facts.csv")

# Вывод первых 5 фактов
print(f"\nПримеры фактов:")
for fact in facts[:5]:
    print(f"  {fact}")

print("\nГотово!")

ОНТОЛОГИЧЕСКАЯ ИНТЕГРАЦИЯ ДАННЫХ О ДИССОВЕТАХ

Источников данных: 8
  Commission: 3 записей (колонки: id, name, code, specialization)
  Specialization: 3 записей (колонки: id, code)
  SpecializationInCommission: 4 записей (колонки: id, commission, specialization)
  Scientist: 4 записей (колонки: id, name, commission, specialization)
  Publication: 4 записей (колонки: id, title, scientist, journal)
  Journal: 4 записей (колонки: id, title)
  JournalSpecialization: 4 записей (колонки: journal, specialization)
  ScientistInCommission: 4 записей (колонки: scientist, commission, specialization)

Всего фактов: 45

Распределение по отношениям:
  включает_специальность: 4
  код: 3
  название: 11
  опубликовал_статью: 4
  опубликована_в: 4
  покрывает_специальность: 4
  представляет_специальность: 4
  состоит_в_совете: 4
  фио: 4
  шифр: 3

✓ Покрытие онтологии: 100% (10/10 отношений)
✓ Сохранено 45 фактов в experiment_data_soviets/generated/all_facts.csv

Примеры фактов:
  ('Scientist_1', 'фио

In [46]:
# ЯЧЕЙКА: Сохранение всех файлов проекта
import json
import csv
import os

# Создаём папку проекта
project_dir = "dissovet_ontology"
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/data", exist_ok=True)

# Сохраняем ячейки как Python-скрипт
with open(f"{project_dir}/module1_integrator.py", 'w', encoding='utf-8') as f:
    f.write("""# Здесь ваш код из ячейки 3 (функции integrate, apply_rule, load_rules, save_to_csv)
""")

# Сохраняем правила
with open(f"{project_dir}/ontology_rules.json", 'w', encoding='utf-8') as f:
    json.dump(rules_json, f, ensure_ascii=False, indent=2)

# Сохраняем CSV-файлы
for name, schema, table in S:
    columns = list(schema.keys())
    with open(f"{project_dir}/data/{name}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(table)

# Сохраняем README.md
with open(f"{project_dir}/README.md", 'w', encoding='utf-8') as f:
    f.write("""# Онтологическая интеграция данных диссертационных советов

Проект выполняет интеграцию гетерогенных данных в унифицированные спецификации фактов (УСФ).

## Структура

- `ontology_rules.json` — правила отображения
- `data/*.csv` — исходные таблицы
- `module1_integrator.py` — модуль интеграции

## Запуск

Скопируйте функции из `module1_integrator.py` в ноутбук.
""")

# Сохраняем .gitignore
with open(f"{project_dir}/.gitignore", 'w') as f:
    f.write("""__pycache__/
*.pyc
.ipynb_checkpoints/
*.db
experiment_data*/
""")

print("✓ Файлы сохранены в dissovet_ontology/")

✓ Файлы сохранены в dissovet_ontology/
